## Data Exploration 

In [ ]:
import os
import glob

root = "data"
version = 'v5'
digital_path = "x/"

folders = ['y9','y10']

for folder in folders:
    files = glob.glob(os.path.join(root, version, folder, "*.wav"))
    
    files.sort()
    print(files)
    
    for i, file_path in enumerate(files, start=1):
        # num = file_path.split("/")[-1].split(".")[0]
        # if int(num) >= 61:
        #     os.remove(file_path)
        new_name = f"{i}.wav" #"digital_"+file_path.split("/")[-1] #
        new_path = os.path.join(root, version, folder, new_name)
        print(file_path, new_path)
        
        # os.rename(file_path, new_path)
    
    print(f"Renamed files in {folder}: {len(files)} files renamed.")


In [ ]:
import os
import glob
import numpy as np
import librosa
from IPython.display import Audio, display
from scipy.io import wavfile


root = "data"
version = 'v3'
digital_path = "x/"
record_low_path = "y1/"
record_high_path = "y2/"

digital_files = glob.glob(os.path.join(root, version, digital_path, "*.wav"))
record_low_files = glob.glob(os.path.join(root, version, record_low_path, "*.wav"))
record_high_files = glob.glob(os.path.join(root, version, record_high_path, "*.wav"))

print(len(digital_files), len(record_low_files), len(record_high_files))


In [ ]:

idx = 11
digital_file = digital_files[idx]
record_low_file = digital_file.replace("x/","y1/")
record_high_file = digital_file.replace("x/","y2/")
print(digital_file, record_low_file, record_high_file)
digital_rate, digital_data = wavfile.read(digital_file)
record_low_rate, record_low_data = wavfile.read(record_low_file)
record_high_rate, record_high_data = wavfile.read(record_high_file)

In [ ]:
x, y = librosa.load(record_low_file, sr=44100, mono=False)
print(x.shape, y)

In [ ]:
print("Playing Digital Signal:")
display(Audio(data=digital_data, rate=digital_rate))

print("Playing Recorded Low-Quality Signal:")
display(Audio(data=record_low_data.T, rate=record_low_rate))

print("Playing Recorded High-Quality Signal:")
display(Audio(data=record_high_data.T, rate=record_high_rate))

In [ ]:
import matplotlib.pyplot as plt

digital_time = [i / digital_rate for i in range(len(digital_data))]
record_low_time = [i / record_low_rate for i in range(len(record_low_data))]
record_high_time = [i / record_high_rate for i in range(len(record_high_data))]


fig, axs = plt.subplots(1, 3, figsize=(16, 4))

axs[0].plot(digital_time, digital_data, color='b')
axs[0].set_title('Digital Signal')
axs[0].set_xlabel('Time (s)')
axs[0].set_ylabel('Amplitude')

axs[1].plot(record_low_time, record_low_data.T[0], color='r')
axs[1].set_title('Recorded Low-Quality Signal')
axs[1].set_xlabel('Time (s)')
axs[1].set_ylabel('Amplitude')

axs[2].plot(record_high_time, record_high_data.T[0], color='g')
axs[2].set_title('Recorded High-Quality Signal')
axs[2].set_xlabel('Time (s)')
axs[2].set_ylabel('Amplitude')

plt.tight_layout()
plt.show()

In [ ]:
from scipy.signal import stft

duration_sec = 10

digital_samples = int(duration_sec * digital_rate)
record_low_samples = int(duration_sec * record_low_rate)
record_high_samples = int(duration_sec * record_high_rate)

digital_data = digital_data[:digital_samples]
record_low_data = record_low_data.T[0][:record_low_samples]
record_high_data = record_high_data.T[0][:record_high_samples]

f1, t1, Zxx1 = stft(digital_data, fs=digital_rate, nperseg=1024)
f2, t2, Zxx2 = stft(record_low_data, fs=record_low_rate, nperseg=1024)
f3, t3, Zxx3 = stft(record_high_data, fs=record_high_rate, nperseg=1024)


fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(21, 6))

ax1.pcolormesh(t1, f1, np.abs(Zxx1), shading='gouraud')
ax1.set_title('Digital Signal Spectrogram (First 10 Seconds)')
ax1.set_xlabel('Time [sec]')
ax1.set_ylabel('Frequency [Hz]')
ax1.set_ylim([0, digital_rate / 2])  

ax2.pcolormesh(t2, f2, np.abs(Zxx2), shading='gouraud')
ax2.set_title('Recorded Low-Quality Signal Spectrogram (First 10 Seconds)')
ax2.set_xlabel('Time [sec]')
ax2.set_ylabel('Frequency [Hz]')
ax2.set_ylim([0, record_low_rate / 2])  

ax3.pcolormesh(t3, f3, np.abs(Zxx3), shading='gouraud')
ax3.set_title('Recorded High-Quality Signal Spectrogram (First 10 Seconds)')
ax3.set_xlabel('Time [sec]')
ax3.set_ylabel('Frequency [Hz]')
ax3.set_ylim([0, record_high_rate / 2])  

plt.tight_layout()
plt.show()

In [ ]:
import os
import librosa
from scipy.io import wavfile
from tqdm import tqdm


def preprocess(digital_path, record_low_path, segment_length=5, stride_length=0.5, target_sampling_rate=16000):
    digital_waveforms = []
    record_low_waveforms = []

    for i in tqdm(range(1, 31)):
        digital_file = os.path.join(digital_path, f"{i}.wav")
        record_low_file = os.path.join(record_low_path, f"{i}.wav")
        
        digital_data, _ = librosa.load(digital_file, sr=target_sampling_rate)
        record_low_data, _ = librosa.load(record_low_file, sr=target_sampling_rate)
        
        segment_samples = int(segment_length * target_sampling_rate)
        stride_samples = int(stride_length * target_sampling_rate)
        
        for start in range(0, len(digital_data) - segment_samples + 1, stride_samples):
            digital_segment = digital_data[start:start + segment_samples]
            record_low_segment = record_low_data[start:start + segment_samples]
            
            digital_waveforms.append(digital_segment)
            record_low_waveforms.append(record_low_segment)
    
    return digital_waveforms, record_low_waveforms


digital_waveforms, record_high_waveforms = preprocess(f"data/{version}/EN_x", f"data/{version}/EN_y2_headphone")

In [ ]:
print(len(digital_waveforms))

In [ ]:
import random

sample_index = random.randint(0, len(digital_waveforms) - 1)

digital_sample = digital_waveforms[sample_index]
record_high_sample = record_high_waveforms[sample_index]

plt.figure(figsize=(12, 5))

plt.subplot(2, 1, 1)
plt.plot(digital_sample)
plt.title("Digital Sample")
plt.xlabel("Time (samples)")
plt.ylabel("Amplitude")

plt.subplot(2, 1, 2)
plt.plot(record_high_sample)
plt.title("Record Low Sample")
plt.xlabel("Time (samples)")
plt.ylabel("Amplitude")

plt.tight_layout()
plt.show()


In [ ]:
print("Playing Digital Sample:")
display(Audio(digital_sample, rate=16000))

print("Playing Record Sample:")
display(Audio(record_high_sample, rate=16000))

## Data Loader

In [ ]:
BATCH_SIZE = 2
NUM_WORKERS = 2
SHUFFLE = True
SAMPLE_RATE = 44100  
SEGMENT_LENGTH = 5
STRIDE_LENGTH = 0.5
STAGE = 1

In [ ]:
import os
import librosa
from tqdm import tqdm


def preprocess(digital_path, record_low_path, segment_length=5, stride_length=0.5, target_sampling_rate=44100, total_files=60, mono=False):
    test_files = 5
    train_val_files = total_files - test_files

    train_end = int(0.8 * train_val_files)
    val_end = train_end + int(0.2 * train_val_files)

    train_digital_waveforms, val_digital_waveforms, test_digital_waveforms = [], [], []
    train_record_low_waveforms, val_record_low_waveforms, test_record_low_waveforms = [], [], []

    for i in tqdm(range(1, total_files + 1)):
        if i <= test_files:
            digital_waveforms = test_digital_waveforms
            record_low_waveforms = test_record_low_waveforms
        elif i <= train_end + test_files:
            digital_waveforms = train_digital_waveforms
            record_low_waveforms = train_record_low_waveforms
        elif i <= val_end + test_files:
            digital_waveforms = val_digital_waveforms
            record_low_waveforms = val_record_low_waveforms

        digital_file = os.path.join(digital_path, f"{i}.wav")
        record_low_file = os.path.join(record_low_path, f"{i}.wav")

        digital_data, _ = librosa.load(digital_file, sr=target_sampling_rate)
        record_low_data, _ = librosa.load(record_low_file, sr=target_sampling_rate, mono=mono)

        segment_samples = int(segment_length * target_sampling_rate)
        stride_samples = int(stride_length * target_sampling_rate)

        for start in range(0, len(digital_data) - segment_samples + 1, stride_samples):
            digital_segment = digital_data[start:start + segment_samples]
            record_low_segment = record_low_data[:, start:start + segment_samples]

            digital_waveforms.append(digital_segment)
            record_low_waveforms.append(record_low_segment)
            
    return (train_digital_waveforms, train_record_low_waveforms), \
           (val_digital_waveforms, val_record_low_waveforms), \
           (test_digital_waveforms, test_record_low_waveforms)


In [ ]:
version = 'v3'
STAGE = 2
DEVICE = 'y1'

(train_digital_waveforms, train_record_low_waveforms), \
(val_digital_waveforms, val_record_low_waveforms), \
(test_digital_waveforms, test_record_low_waveforms) = preprocess(f"data/{version}/x", f"data/{version}/{DEVICE}", total_files=20)
print(len(train_record_low_waveforms), len(val_digital_waveforms), len(test_digital_waveforms))


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader


class EqualizerDataset(Dataset):
    def __init__(self, digital_waveforms, record_target_waveforms, speakers=None, return_dict=False):
        assert len(digital_waveforms) == len(record_target_waveforms), \
            "Input and output waveforms lists must be of the same length"

        self.digital_waveforms = digital_waveforms
        self.record_target_waveforms = record_target_waveforms
        self.speaker_list = speakers
        self.return_dict = return_dict

    def __len__(self):
        return len(self.digital_waveforms)

    def __getitem__(self, idx):
        digital_sample = torch.tensor(self.digital_waveforms[idx], dtype=torch.float32)
        record_target_sample = torch.tensor(self.record_target_waveforms[idx], dtype=torch.float32)
        try:
            speaker = self.speaker_list[idx]
            text_emb = torch.load(f"embeddings/{speaker}/{speaker}_1.pt")[0].cpu()
        except:
            text_emb = None

        if self.return_dict:
            return {'input_values': digital_sample, 'labels': record_target_sample, "text_emb": text_emb}

        return digital_sample, record_target_sample, text_emb
    

In [ ]:
train_dataset = EqualizerDataset(train_digital_waveforms, train_record_low_waveforms)
val_dataset = EqualizerDataset(val_digital_waveforms, val_record_low_waveforms)
test_dataset = EqualizerDataset(test_digital_waveforms, test_record_low_waveforms)

In [ ]:
for digital_batch, record_low_batch, emb_batch in tqdm(train_dataset):
    print("Digital Batch Shape:", digital_batch.shape)
    print("Record Low Batch Shape:", record_low_batch.shape)

    break


## Model

### Demuc

In [ ]:
from IPython import display as disp
import torch
import torchaudio
from denoiser import pretrained
from denoiser.dsp import convert_audio
import librosa

model = pretrained.dns64().cpu()
wav, _ = librosa.load("data/EN_x/0.wav", sr=model.sample_rate)
with torch.no_grad():
    denoised = model(torch.FloatTensor(wav[None]))[0]
disp.display(disp.Audio(wav.data, rate=model.sample_rate))
disp.display(disp.Audio(denoised.data, rate=model.sample_rate))

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params}")


## Trainer

In [ ]:
from transformers import Trainer, TrainingArguments
import torch
from torch.utils.data import Dataset
import torch.nn as nn
from loss import STFTLoss


stft_loss_fn = STFTLoss()

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions
    mse = ((preds - labels) ** 2).mean()
    stft_loss = stft_loss_fn(preds, labels)
    
    return {"mse": mse, "stft_loss": stft_loss}


class TrainerModelWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.mse_loss_fn = nn.MSELoss()  

    def forward(self, input_values, labels=None):
        outputs = self.model(input_values)
        loss = self.mse_loss_fn(outputs, labels) + stft_loss_fn(outputs, labels) if labels is not None else None
        
        return (loss, outputs) if loss is not None else outputs


In [ ]:
training_args = TrainingArguments(
    output_dir="assets",
    eval_strategy="epoch",
    learning_rate=5e-5,
    save_strategy='epoch',
    logging_steps=4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=10,
    weight_decay=0.01,
    logging_dir="assets/logs",
    report_to="wandb",
    save_safetensors=False,
)

model = TrainerModelWrapper(Wav2VecEqualizer(freeze_encoder=False))

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

# trainer.train()

### Stage 2

In [ ]:
import torch
from IPython.display import Audio
import random
from safetensors.torch import load_file
from models.demucs_equalizer import DemucsEqualizer, DoubleDemucsEqualizer

STAGE = 2

(train_digital_waveforms, train_record_low_waveforms), \
(val_digital_waveforms, val_record_low_waveforms), \
(test_digital_waveforms, test_record_low_waveforms) = preprocess("data/EN_x", f"data/EN_y{STAGE}_earphone")

train_dataset = EqualizerDataset(train_digital_waveforms, train_record_low_waveforms, return_dict=True)
val_dataset = EqualizerDataset(val_digital_waveforms, val_record_low_waveforms, return_dict=True)
test_dataset = EqualizerDataset(test_digital_waveforms, test_record_low_waveforms, return_dict=True)


In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint_path = "assets/stage2-earphone/pytorch_model.bin" # assets/stage2-headphone/pytorch_model.bin

model = DoubleDemucsEqualizer("assets/stage1/pytorch_model.bin", device=device)
state_dict = torch.load(checkpoint_path, map_location=device)
state_dict = {k[6:]: v for k, v in state_dict.items()}

model.load_state_dict(state_dict)
model.eval()  

model = model.to(device)

sample_idx = random.randint(500, 700)
print(sample_idx) # 767 586 598
sample = test_dataset[sample_idx]
y1_sample = test_dataset_stage_1[sample_idx]

input_waveform = sample['input_values'].unsqueeze(0).unsqueeze(0)
target_waveform = sample['labels'].numpy()  

with torch.no_grad():
    first_waveform = model.model2(input_waveform).squeeze(0)
    second_waveform = model.model1(first_waveform).squeeze(0).squeeze(0).cpu().numpy()

print("Original Digital Audio:")
display(Audio(input_waveform.squeeze().numpy(), rate=16000)) 

print("Y1 Label:")
display(Audio(y1_sample['labels'].reshape(-1), rate=16000)) 

print("Target Audio:")
display(Audio(target_waveform, rate=16000))

print("First Audio:")
display(Audio(first_waveform, rate=16000))

print("Second Audio:")
display(Audio(second_waveform, rate=16000))


## Evaluation

In [ ]:
'''
{
    "x": "Digital Input",              x
    "y1": "Fransun E20",               e2
    "y2": "Airline Earphone",          e1
    "y3": "JBL C100SI",                e3
    "y4": "Sennheiser IE 300",         e5
    "y5": "Sony WF-1000XM5",           h4
    "y6": "Zihnic Foldable",           h3
    "y7": "Aurvana Air",               e4
    "y8": "Beyerdynamic DT700",        h5
    "y9": "Realme RM66",               h1
    "y10": "P2961",                    h2
}
'''

# This is the test for the curve, so we only report single model, single pair (e3, e5)

#  (train + val + test) varies: 10 15 20 30 40
#  (test) fixed, first 5 files

# ==> meaning (train + val): 5 10 15 25 35. The corresponding checkpoint is below:

checkpoint_mapping = {
    'y3-y8-5': ["", ""],
    'y3-y8-10': ["", ""],
    'y3-y8-15': ["", ""],
    'y3-y8-25': ["", ""],
    'y3-y8-35': ["", ""],
}

### Stage 1

In [ ]:
import torch
from IPython.display import Audio
import random
from safetensors.torch import load_file
from models.demucs_equalizer import DemucsEqualizer, StyleTransform2
from safetensors.torch import load_file
from utils import preprocess 
from dataset import EqualizerDataset

STAGE = 1
version = 'v5'
setting = 'y3-y8-5' ####### NOTE: 5 10 15 25 35 #######
poor_speaker = f"data/{version}/{setting.split('-')[0]}"

(train_digital_waveforms, train_record_low_waveforms), \
(val_digital_waveforms, val_record_low_waveforms), \
(test_digital_waveforms, test_record_low_waveforms) = preprocess(f"data/{version}/x", poor_speaker, target_sampling_rate=44100, total_files=6)

test_dataset_stage_1 = EqualizerDataset(test_digital_waveforms, test_record_low_waveforms, speakers=None, return_dict=True)

100%|██████████| 10/10 [00:03<00:00,  2.84it/s]


In [ ]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint_path = checkpoint_mapping[setting][0]

model = DemucsEqualizer(device=device) 
state_dict = torch.load(checkpoint_path, map_location=device)
state_dict = {k[6:]: v for k, v in state_dict.items()}
model.load_state_dict(state_dict)
model.eval()  

model = model.to(device)

sample_idx = random.randint(0, len(test_dataset_stage_1) - 1) 
print(sample_idx)

sample = test_dataset_stage_1[sample_idx]

input_waveform = sample['input_values'].unsqueeze(0)
target_waveform = sample['labels'].numpy()  

with torch.no_grad():
    enhanced_waveform = model(input_waveform)
    enhanced_waveform = enhanced_waveform.squeeze(0).squeeze(0).cpu().numpy()

print("Original Digital Audio:")
display(Audio(input_waveform.squeeze().numpy(), rate=44100)) 

print("Target Audio:")
display(Audio(target_waveform, rate=44100))

print("Output Audio:")
display(Audio(enhanced_waveform, rate=44100))


### Stage 2

In [ ]:
import torch
from IPython.display import Audio
import random
from safetensors.torch import load_file
from models.demucs_equalizer import DemucsEqualizer, DoubleDemucsEqualizer

STAGE = 2
VERSION = 'v5' 

(train_digital_waveforms, train_record_low_waveforms), \
(val_digital_waveforms, val_record_low_waveforms), \
(test_digital_waveforms, test_record_low_waveforms) = preprocess(f"data/{VERSION}/x", f"data/{VERSION}/{speaker.split('-')[1]}", target_sampling_rate=44100, total_files=6)

train_dataset = EqualizerDataset(train_digital_waveforms, train_record_low_waveforms, return_dict=True)
val_dataset = EqualizerDataset(val_digital_waveforms, val_record_low_waveforms, return_dict=True)
test_dataset = EqualizerDataset(test_digital_waveforms, test_record_low_waveforms, return_dict=True)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint_path = checkpoint_mapping[speaker][1]

model = DoubleDemucsEqualizer(checkpoint_mapping[speaker][0], model_name='v1', device=device)
state_dict = torch.load(checkpoint_path, map_location=device)
state_dict = {k[6:]: v for k, v in state_dict.items()}
model.load_state_dict(state_dict)
model.eval()  

model = model.to(device)

sample_idx = random.randint(0, len(test_dataset_stage_1) - 1) #  2322 2423
print(sample_idx)
 
sample = test_dataset[sample_idx]
y1_sample = test_dataset_stage_1[sample_idx]

input_waveform = sample['input_values'].unsqueeze(0)
if len(input_waveform.shape) == 2:
    input_waveform = input_waveform.unsqueeze(1)
target_waveform = sample['labels'].numpy()  
input_waveform = torch.cat([input_waveform, input_waveform], dim=1)        

with torch.no_grad():
    first_waveform = model.model2(input_waveform)
    first_waveform = first_waveform.mean(dim=1)[:,0,:]  
    first_waveform = first_waveform.reshape(first_waveform.shape[0], -1)
    second_waveform = model.model1(first_waveform)
    second_waveform = second_waveform.squeeze(0).cpu().numpy()

print("Original Digital Audio:")
display(Audio(input_waveform.squeeze().numpy(), rate=44100)) 

print("Y1 Label:")
display(Audio(y1_sample['labels'], rate=44100)) 

print("Target Audio:")
display(Audio(target_waveform, rate=44100))

print("First Audio:")
display(Audio(first_waveform, rate=44100))

print("Second Audio:")
display(Audio(second_waveform, rate=44100))
